# Ejercicio: Extracción de datos de la API de TMDB (The Movie Database)

![imagen](https://www.themoviedb.org/assets/2/v4/logos/v2/blue_square_2-d537fb228cf3ded904ef09b136fe3fec72548ebc1fea3fbbd1ad9e36364db38b.svg)

Trabajaremos con **TMDB**, una de las bases de datos de cine más importantes del mundo. El objetivo es extraer información de películas para realizar un análisis de mercado inicial.

### 1. Registro y Documentación
Tendrás que [registrarte en TMDB](https://www.themoviedb.org/signup) para obtener tu **API Key (v3 auth)** y consultar la [documentación oficial](https://developer.themoviedb.org/reference/intro/getting-started).

### 2. Objetivo del Ejercicio
Queremos que consultes la API para que te devuelva la información de las películas que empiecen por la **inicial de tu nombre** (parámetro `query`). 

Debes guardar la información en un archivo `.csv` con la siguiente estructura de columnas:

| Columna | Descripción |
| :--- | :--- |
| **id** | ID interno de la película en TMDB |
| **title** | Título de la película |
| **release_date** | Fecha de estreno |
| **genres** | Nombres de los géneros (ej: "Acción, Comedia") |
| **vote_average** | Puntuación media de los usuarios |
| **overview** | Sinopsis o resumen de la trama |

### 3. El Reto: Mapeo de Géneros
A diferencia de otras APIs, el endpoint de búsqueda de películas devuelve los géneros como una lista de IDs numéricos (ej: `[28, 12]`). 

**Tu labor es:**
1. Consultar el endpoint de "Genre List" para obtener la relación entre IDs y nombres.
2. Sustituir los IDs en tu DataFrame final por los nombres reales de los géneros (separados por comas).

---

### Paso 0 — Importamos librerías y configuramos la API key

In [1]:
import requests
import pandas as pd

# Nuestra API key personal (la obtenemos en themoviedb.org > perfil > Settings > API)
api_key = "0fe00c5b9213c33d4dc3192bb57ae2fc"  # <-- SUSTITUIR por tu key real

# La URL base de la API: todos los endpoints empiezan por aquí
url_base = "https://api.themoviedb.org/3"

# Nuestra inicial: A de Andoni
mi_inicial = "A"

### Paso 1 — Obtener el diccionario de géneros (ID → Nombre)

La API de TMDB nos da los géneros como IDs numéricos (28, 35, 12...).  
Necesitamos pedirle la "tabla de traducción" para saber qué nombre corresponde a cada ID.  
El endpoint es: `/genre/movie/list`

En **Postman** lo probaríamos así:  
`GET https://api.themoviedb.org/3/genre/movie/list?api_key=TU_KEY&language=es`

In [2]:
def get_genres_map(api_key):
    """
    Pedimos a la API la lista completa de géneros.
    Devolvemos un diccionario {id: nombre} para poder traducir después.
    """
    # Construimos la URL del endpoint de géneros
    url = f"{url_base}/genre/movie/list"
    
    # Los parámetros los pasamos como diccionario: requests los añade a la URL automáticamente
    # Esto es equivalente a escribir ?api_key=xxx&language=es al final de la URL
    params = {
        "api_key": api_key,
        "language": "es"       # Pedimos los nombres en español
    }
    
    # Hacemos la petición
    response = requests.get(url, params=params)
    data = response.json()
    
    # data["genres"] es una lista de diccionarios: [{"id": 28, "name": "Acción"}, ...]
    # Lo convertimos en un diccionario simple: {28: "Acción", 35: "Comedia", ...}
    genres_map = {}
    for genre in data["genres"]:
        genres_map[genre["id"]] = genre["name"]
    
    return genres_map

# Ejecutamos la función y vemos el resultado
genres_map = get_genres_map(api_key)
print(f"Tenemos {len(genres_map)} géneros")
print()
print(genres_map)

Tenemos 19 géneros

{28: 'Acción', 12: 'Aventura', 16: 'Animación', 35: 'Comedia', 80: 'Crimen', 99: 'Documental', 18: 'Drama', 10751: 'Familia', 14: 'Fantasía', 36: 'Historia', 27: 'Terror', 10402: 'Música', 9648: 'Misterio', 10749: 'Romance', 878: 'Ciencia ficción', 10770: 'Película de TV', 53: 'Suspense', 10752: 'Bélica', 37: 'Western'}


### Paso 2 — Buscar películas por nuestra inicial

El endpoint de búsqueda es: `/search/movie`  
Le pasamos `query=A` y nos devuelve películas cuyo título empiece o contenga "A".

**El tema de la paginación:**  
La API nos da máximo 20 resultados por página. Si hay 500 películas, habrá 25 páginas.  
Tenemos que ir pidiendo `page=1`, `page=2`... hasta llegar al total.  
La propia respuesta nos dice cuántas páginas hay en el campo `total_pages`.

En **Postman** lo probaríamos así:  
`GET https://api.themoviedb.org/3/search/movie?api_key=TU_KEY&query=A&language=es&page=1`

In [3]:
def search_movies(api_key, query, max_pages=5):
    """
    Buscamos películas por la inicial.
    Recorremos varias páginas para obtener más de 20 resultados.
    Limitamos a max_pages páginas para no hacer demasiadas peticiones.
    """
    # Aquí iremos guardando todas las películas de todas las páginas
    todas_las_peliculas = []
    
    # Empezamos por la página 1
    page = 1
    
    while page <= max_pages:
        # Construimos la petición para esta página
        url = f"{url_base}/search/movie"
        params = {
            "api_key": api_key,
            "query": query,
            "language": "es",
            "page": page
        }
        
        response = requests.get(url, params=params)
        data = response.json()
        
        # Sacamos las películas de esta página y las añadimos a nuestra lista
        resultados = data["results"]
        todas_las_peliculas.extend(resultados)
        
        # Si estamos en la primera página, mostramos cuántas páginas hay en total
        if page == 1:
            total_pages = data["total_pages"]
            total_results = data["total_results"]
            print(f"La API ha encontrado {total_results} películas en {total_pages} páginas")
            print(f"Nosotros vamos a descargar {min(max_pages, total_pages)} páginas")
        
        # Si ya no hay más páginas, paramos
        if page >= data["total_pages"]:
            break
        
        page += 1
    
    print(f"Descargadas {len(todas_las_peliculas)} películas en total")
    return todas_las_peliculas

# Ejecutamos la búsqueda
peliculas_raw = search_movies(api_key, mi_inicial, max_pages=5)

La API ha encontrado 10000 películas en 500 páginas
Nosotros vamos a descargar 5 páginas
Descargadas 100 películas en total


In [4]:
# Echamos un vistazo a la primera película para entender la estructura
# Así vemos qué claves tiene cada película y dónde están los genre_ids
peliculas_raw[0]

{'adult': False,
 'backdrop_path': '/sdZSjtGUTSN8B3al5o0f2WoQfQQ.jpg',
 'genre_ids': [878, 12, 14],
 'id': 83533,
 'original_language': 'en',
 'original_title': 'Avatar: Fire and Ash',
 'overview': "Jake Sully y Neytiri enfrentan una nueva amenaza en Pandora: los Ash People, una tribu Na'vi violenta y sedienta de poder, liderada por la implacable Varang. Tras la devastadora guerra contra la RDA y la pérdida de su hijo mayor, la familia de Jake deberá luchar por su supervivencia y el futuro de Pandora en un conflicto que llevará a los personajes a sus límites emocionales y físicos. Con nuevos y antiguos aliados, esta épica visual y emocional redefine el destino de un mundo al borde del abismo.",
 'popularity': 329.536,
 'poster_path': '/4n1U0Mwn7djux6VKNYDRWPgS2x6.jpg',
 'release_date': '2025-12-17',
 'title': 'Avatar: Fuego y ceniza',
 'video': False,
 'vote_average': 7.267,
 'vote_count': 1921}

### Paso 3 — Crear el DataFrame y mapear los géneros

Ahora juntamos todo:  
1. Creamos el DataFrame con las columnas que nos piden  
2. Traducimos los `genre_ids` (números) a nombres usando el diccionario del Paso 1

In [5]:
# Creamos una lista de diccionarios "limpios" con solo las columnas que necesitamos
peliculas_limpias = []

for peli in peliculas_raw:
    
    # Traducimos los IDs de género a nombres
    # Para cada ID en genre_ids, buscamos su nombre en nuestro diccionario genres_map
    # Si un ID no existe en el diccionario (raro pero posible), ponemos "Desconocido"
    nombres_generos = []
    for genre_id in peli["genre_ids"]:
        nombre = genres_map.get(genre_id, "Desconocido")
        nombres_generos.append(nombre)
    
    # Unimos los nombres con comas: ["Acción", "Aventura"] -> "Acción, Aventura"
    generos_texto = ", ".join(nombres_generos)
    
    # Creamos el diccionario limpio para esta película
    peliculas_limpias.append({
        "id": peli["id"],
        "title": peli["title"],
        "release_date": peli.get("release_date", ""),  # Usamos .get() por si alguna no tiene fecha
        "genres": generos_texto,
        "vote_average": peli["vote_average"],
        "overview": peli.get("overview", "")
    })

# Creamos el DataFrame
df = pd.DataFrame(peliculas_limpias)

print(f"DataFrame con {len(df)} películas y {len(df.columns)} columnas")
print(f"Columnas: {list(df.columns)}")
df.head(10)

DataFrame con 100 películas y 6 columnas
Columnas: ['id', 'title', 'release_date', 'genres', 'vote_average', 'overview']


,id,title,release_date,genres,vote_average,overview
0,83533,Avatar: Fuego y ceniza,2025-12-17,"Ciencia ficción, Aventura, Fantasía",7.267,Jake Sully y Neytiri enfrentan una nueva amena...
1,1301306,A Woman Scorned,2025-06-09,Acción,6.464,
2,1634858,Cuerno Azulado,2025-09-10,Acción,6.706,Un ex narcotraficante busca reivindicarse como...
3,1310568,Asesinato en la Embajada,2025-11-14,"Misterio, Suspense, Acción",5.803,"1934. Miranda Green, detective privada, invest..."
4,1054867,Una batalla tras otra,2025-09-23,"Suspense, Crimen, Comedia",7.389,"Un ex revolucionario, tras años apartado de la..."
5,277218,A,1965-01-02,Animación,6.300,
6,77165,Y Dios le dijo a Caín,1970-02-05,"Western, Terror, Misterio",6.700,"El teniente Hamilton, del ejército nordista, q..."
7,166680,Avalancha,1978-09-29,"Acción, Aventura, Drama",3.600,Unos turistas luchan por sobrevivir después de...
8,1368166,La asistenta,2025-12-18,"Misterio, Suspense",7.213,"Una joven (Sydney Sweeney), con un pasado comp..."
9,40817,Baciami ancora,2010-01-29,Comedia,5.800,Carlo y Giulia se han separado y están a la es...


### Paso 4 — Exportar a CSV

In [6]:
# Exportamos el DataFrame a un archivo CSV
# index=False para que no añada una columna extra con el índice de Pandas
df.to_csv("peliculas_TMDB_A.csv", index=False)

print("Archivo 'peliculas_TMDB_A.csv' guardado correctamente")

Archivo 'peliculas_TMDB_A.csv' guardado correctamente
